# Day 7: Mini-Project - Bookmark Manager

- Full CRUD REST API with FastAPI
- SQLAlchemy models
- Pydantic schemas
- Pagination and search

## Project Overview

Build a bookmark manager API with Bookmark model (id, title, url, description, created_at). GET/POST /bookmarks, GET/PUT/DELETE /bookmarks/{id}.

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Text, DateTime
from sqlalchemy.orm import sessionmaker, declarative_base
from datetime import datetime

Base = declarative_base()
engine = create_engine("sqlite:///bookmarks.db", connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(bind=engine)

class Bookmark(Base):
    __tablename__ = "bookmarks"
    id = Column(Integer, primary_key=True, index=True)
    title = Column(String(200), nullable=False)
    url = Column(String(500), nullable=False)
    description = Column(Text)
    created_at = Column(DateTime, default=datetime.utcnow)

Base.metadata.create_all(engine)
print("Bookmark model created")

In [ ]:
from fastapi import FastAPI, Depends, HTTPException
from sqlalchemy.orm import Session
from pydantic import BaseModel

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

class BookmarkCreate(BaseModel):
    title: str
    url: str
    description: str | None = None

class BookmarkOut(BaseModel):
    id: int
    title: str
    url: str
    description: str | None
    created_at: datetime
    class Config:
        from_attributes = True

app = FastAPI()

@app.get("/bookmarks", response_model=list[BookmarkOut])
def list_bookmarks(skip: int = 0, limit: int = 20, q: str | None = None, db: Session = Depends(get_db)):
    query = db.query(Bookmark)
    if q:
        query = query.filter(Bookmark.title.contains(q))
    return query.offset(skip).limit(limit).all()

@app.post("/bookmarks", response_model=BookmarkOut, status_code=201)
def create_bookmark(b: BookmarkCreate, db: Session = Depends(get_db)):
    bookmark = Bookmark(title=b.title, url=b.url, description=b.description)
    db.add(bookmark)
    db.commit()
    db.refresh(bookmark)
    return bookmark

print("List with pagination, Create")

In [ ]:
@app.get("/bookmarks/{id}", response_model=BookmarkOut)
def get_bookmark(id: int, db: Session = Depends(get_db)):
    b = db.query(Bookmark).filter(Bookmark.id == id).first()
    if not b:
        raise HTTPException(404, "Not found")
    return b

@app.delete("/bookmarks/{id}", status_code=204)
def delete_bookmark(id: int, db: Session = Depends(get_db)):
    b = db.query(Bookmark).filter(Bookmark.id == id).first()
    if not b:
        raise HTTPException(404, "Not found")
    db.delete(b)
    db.commit()
    return None

print("Get, Delete. Run: uvicorn main:app --reload")

## Key Takeaways

- FastAPI + SQLAlchemy + Pydantic
- get_db dependency, HTTPException for 404
- Pagination and search

## Week 15 Complete!

Next week: SQLite, PostgreSQL, Redis, Docker, and CI/CD!